# MIT4 Mitral Cell Model

Interactive NEURON simulation of a 4-compartment mitral cell from **ModelDB 2487**.

Based on: Bhalla & Bower (1993) *J. Neurophysiol.* 69:1948-1983
Model implementation by Andrew Davison, The Babraham Institute.

**Compartments:** soma, glomerulus, primary dendrite, secondary dendrite + 3 linking compartments

In [20]:
from neuron import h
from neuron.units import ms, mV
import numpy as np
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 120
%matplotlib inline

In [21]:
# Load compiled mechanisms
mech_path = os.path.abspath('2487-master/arm64/libnrnmech.dylib')
h.nrn_load_dll(mech_path)
print('Mechanisms loaded successfully')

Mechanisms loaded successfully


## Cell Construction
The MIT4 model has 7 sections in a tree topology:

In [22]:
def build_mit4(n_secondary=4):
    """Build and return the MIT4 mitral cell model with multiple secondary dendrites."""

    # ---------------- Parameters ----------------
    Atotal = 100000.0
    Len = 100.0
    RM = 100000.0
    Erest = -65.0

    p, q, r = 0.051, 0.084, 0.328

    gpg = 5.86e-5
    gsp = 5.47e-5
    gsd = 1.94e-4

    # ---------------- Create sections ----------------
    sections = {}

    for name in ["soma", "glom", "prim", "s2p", "p2g"]:
        sections[name] = h.Section(name=name)

    sections["dend"] = []
    sections["s2d"] = []

    for i in range(n_secondary):
        sections["dend"].append(h.Section(name=f"dend_{i}"))
        sections["s2d"].append(h.Section(name=f"s2d_{i}"))

    soma = sections["soma"]
    glom = sections["glom"]
    prim = sections["prim"]
    s2p = sections["s2p"]
    p2g = sections["p2g"]

    # ---------------- Topology ----------------
    s2p.connect(soma(0), 0)
    prim.connect(s2p(1), 0)
    p2g.connect(prim(1), 0)
    glom.connect(p2g(1), 0)

    for s2d, dend in zip(sections["s2d"], sections["dend"]):
        s2d.connect(soma(1), 0)
        dend.connect(s2d(1), 0)

    for sec in sections["s2d"]:
        sec.L = 1
        sec.diam = 1

    for sec in [s2p, p2g]:
        sec.L = 1
        sec.diam = 1

    # ---------------- Areas ----------------
    Asoma = p * Atotal
    Aglom = q * Atotal
    Aprim = r * Atotal
    Adend = Atotal - Asoma - Aglom - Aprim

    area_each = Adend / n_secondary

    # ---------------- Helper functions ----------------
    def set_size(sec, area):
        sec.diam = area / (np.pi * Len)
        sec.L = Len

    def set_ra(sec, g):
        sec.Ra = (np.pi * 1e4) / (4 * Atotal) * (1.0 / g)

    def setup(sec, area, channels):
        sec.Ra = 1e-7
        set_size(sec, area)

        for ch in channels:
            sec.insert(ch)

        sec.e_pas = Erest
        sec.g_pas = 1.0 / RM

    # ---------------- Soma ----------------
    setup(
        soma,
        Asoma,
        [
            "pas",
            "nafast",
            "kfasttab",
            "kslowtab",
            "kA",
            "kca3",
            "lcafixed",
            "cad",
        ],
    )

    soma.gnabar_nafast = 0.1532
    soma.gkbar_kfasttab = 0.1956
    soma.gkbar_kslowtab = 0.0028
    soma.gkbar_kA = 0.00587
    soma.gkbar_kca3 = 0.0142
    soma.gcabar_lcafixed = 0.0040
    soma.depth_cad = 8

    # ---------------- Glomerulus ----------------
    setup(
        glom,
        Aglom,
        [
            "pas",
            "kslowtab",
            "lcafixed",
            "cad",
        ],
    )

    glom.gkbar_kslowtab = 0.020
    glom.gcabar_lcafixed = 0.0095
    glom.depth_cad = 8

    # ---------------- Primary dendrite ----------------
    setup(
        prim,
        Aprim,
        [
            "pas",
            "nafast",
            "kfasttab",
            "kslowtab",
            "lcafixed",
            "cad",
        ],
    )

    prim.gkbar_kfasttab = 0.00123
    prim.gnabar_nafast = 0.00134
    prim.gkbar_kslowtab = 0.00174
    prim.gcabar_lcafixed = 0.0022
    prim.depth_cad = 8

    # ---------------- Secondary dendrites ----------------
    for dend in sections["dend"]:

        setup(
            dend,
            area_each,
            [
                "pas",
                "kfasttab",
                "nafast",
            ],
        )

        dend.gkbar_kfasttab = 0.0330
        dend.gnabar_nafast = 0.0226

    # ---------------- Axial resistances ----------------
    for sec in sections["s2d"]:
        set_ra(sec, gsd)

    set_ra(s2p, gsp)
    set_ra(p2g, gpg)

    # ---------------- Reversal potentials ----------------
    for sec in h.allsec():

        if sec.has_membrane("ca_ion"):
            sec.eca = 70
            sec.cai = 1e-5
            sec.cao = 2

        if sec.has_membrane("na_ion"):
            sec.ena = 45

        if sec.has_membrane("k_ion"):
            sec.ek = -70

    return sections


print("Cell builder defined.")

Cell builder defined.


In [23]:
import neuron
print(neuron.__file__)
print(neuron.__version__)

c:\nrn\lib\python\neuron\__init__.py
9.0.1


In [24]:
from neuron import h
print(h.nrnversion())

NEURON -- VERSION 9.0.1 HEAD (b12a541+) 2025-11-14


In [25]:
import os
print(os.listdir("x86_64"))

['cadecay.cpp', 'cadecay.o', 'kA.cpp', 'kA.o', 'kca3.cpp', 'kca3.o', 'kfasttab.cpp', 'kfasttab.o', 'kslowtab.cpp', 'kslowtab.o', 'lcafixed.cpp', 'lcafixed.o', 'libnrnmech.so', 'makemod2c_inc', 'mod_func.cpp', 'mod_func.o', 'nafast.cpp', 'nafast.o', 'special', 'special.nrn']


In [26]:
from neuron import h

s = h.Section()
s.insert("nafast")
print("nafast loaded successfully!")

nafast loaded successfully!


In [27]:
import os

print("Current directory:", os.getcwd())
print("Current dir files:", "nrnmech.dll" in os.listdir())

print("x86_64 exists:", os.path.exists("x86_64"))
if os.path.exists("x86_64"):
    print("x86_64 contents:")
    print(os.listdir("x86_64"))

Current directory: e:\IISER CAMP 2026\Code\Project_2\2487-master
Current dir files: True
x86_64 exists: True
x86_64 contents:
['cadecay.cpp', 'cadecay.o', 'kA.cpp', 'kA.o', 'kca3.cpp', 'kca3.o', 'kfasttab.cpp', 'kfasttab.o', 'kslowtab.cpp', 'kslowtab.o', 'lcafixed.cpp', 'lcafixed.o', 'libnrnmech.so', 'makemod2c_inc', 'mod_func.cpp', 'mod_func.o', 'nafast.cpp', 'nafast.o', 'special', 'special.nrn']


In [28]:
from neuron import h
import os

dll = os.path.join(os.getcwd(), "nrnmech.dll")
print("DLL path:", dll)

try:
    h.nrn_load_dll(dll)
    print("DLL loaded successfully")
except Exception as e:
    print("Load error:", e)

NEURON: The user defined name already exists: cad
 near line 0
 ^
        nrn_load_dll("e:\IISER C...")


DLL path: e:\IISER CAMP 2026\Code\Project_2\2487-master\nrnmech.dll
Load error: hocobj_call error: hoc_execerror: The user defined name already exists: cad


In [29]:
from neuron import h

s = h.Section()

try:
    s.insert("nafast")
    print("SUCCESS: nafast is available")
except Exception as e:
    print("ERROR:", e)

SUCCESS: nafast is available


In [30]:
import os

for f in os.listdir("x86_64"):
    if "nrn" in f.lower() or f.endswith(".dll") or f.endswith(".so"):
        print(f)

libnrnmech.so
special.nrn


In [31]:
# Build the cell with 4 secondary dendrites
cell = build_mit4(n_secondary=4)

print("Cell built! Topology:")
h.topology()

Cell built! Topology:

|-|       __nrnsec_00000178ad13f100(0-1)
|-|       soma(0-1)
   `|       s2d_0(0-1)
     `|       dend_0(0-1)
   `|       s2d_1(0-1)
     `|       dend_1(0-1)
   `|       s2d_2(0-1)
     `|       dend_2(0-1)
   `|       s2d_3(0-1)
     `|       dend_3(0-1)
 `|       s2p(0-1)
   `|       prim(0-1)
     `|       p2g(0-1)
       `|       glom(0-1)



1.0

## Interactive Simulation Controls

Adjust the stimulation parameters and click **Run Simulation** to see the response.

In [32]:
def run_mit4(Ifull, stim_location, stim_dur, stim_delay, tstop,
             n_secondary=4):
    """
    Run MIT4 simulation.
    Records only the first secondary dendrite for plotting.
    """

    # Build cell
    cell = build_mit4(n_secondary=n_secondary)

    # ---------------- Stimulation ----------------
    alphas = 1.37
    alphag = 1.85
    Atotal = 100000.0

    injcurrdens = Ifull / 100072.0

    if stim_location in ["Soma", "Both"]:
        sstim = h.IClamp(cell["soma"](0.5))
        sstim.delay = stim_delay
        sstim.dur = stim_dur
        sstim.amp = alphas * injcurrdens * Atotal

    if stim_location in ["Glomerulus", "Both"]:
        gstim = h.IClamp(cell["glom"](0.5))
        gstim.delay = stim_delay
        gstim.dur = stim_dur
        gstim.amp = alphag * injcurrdens * Atotal

    # ---------------- Record ----------------
    rec = {}
    rec["t"] = h.Vector().record(h._ref_t)
    rec["soma"] = h.Vector().record(cell["soma"](0.5)._ref_v)
    rec["glom"] = h.Vector().record(cell["glom"](0.5)._ref_v)
    rec["prim"] = h.Vector().record(cell["prim"](0.5)._ref_v)

    # Record ONLY the first secondary dendrite
    rec["dend"] = h.Vector().record(cell["dend"][0](0.5)._ref_v)

    # ---------------- Run ----------------
    h.dt = 0.025
    h.finitialize(-65)

    while h.t < tstop:
        h.fadvance()

    # ---------------- Convert ----------------
    result = {}
    result["t"] = np.array(rec["t"])
    result["soma"] = np.array(rec["soma"])
    result["glom"] = np.array(rec["glom"])
    result["prim"] = np.array(rec["prim"])
    result["dend"] = np.array(rec["dend"])

    return result

In [33]:
# Create interactive controls
stim_loc = widgets.Dropdown(
    options=['Soma', 'Glomerulus', 'Both'],
    value='Soma',
    description='Stimulus at:'
)

Ifull_slider = widgets.FloatSlider(
    value=1.6,
    min=0.0,
    max=3.0,
    step=0.1,
    description='Ifull (nA):',
    readout_format='.2f'
)

stim_dur = widgets.FloatSlider(
    value=100,
    min=5,
    max=500,
    step=5,
    description='Duration (ms):'
)

stim_delay = widgets.FloatSlider(
    value=50,
    min=0,
    max=200,
    step=5,
    description='Delay (ms):'
)

tstop_slider = widgets.FloatSlider(
    value=200,
    min=20,
    max=500,
    step=10,
    description='Tstop (ms):'
)

run_btn = widgets.Button(
    description='Run Simulation',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

out = widgets.Output()


def on_run(b):

    with out:

        clear_output(wait=True)
        print("Running simulation...")

        try:

            result = run_mit4(
                Ifull=Ifull_slider.value,
                stim_location=stim_loc.value,
                stim_dur=stim_dur.value,
                stim_delay=stim_delay.value,
                tstop=tstop_slider.value,
                n_secondary=8      # Change this to any number
            )

            fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

            colors = {
                "soma": "#1f77b4",
                "glom": "#ff7f0e",
                "prim": "#2ca02c",
                "dend": "#d62728"
            }

            # ---------------- Top panel ----------------
            ax = axes[0]

            ax.plot(result["t"], result["soma"], label="Soma", color=colors["soma"])
            ax.plot(result["t"], result["glom"], label="Glomerulus", color=colors["glom"])
            ax.plot(result["t"], result["prim"], label="Primary", color=colors["prim"])
            ax.plot(result["t"], result["dend"], label="Secondary", color=colors["dend"])

            ax.axhline(0, color="gray", linestyle=":", linewidth=0.5)

            ax.set_ylabel("Voltage (mV)")
            ax.set_title(
                f"MIT4 Mitral Cell ({8} Secondary Dendrites)"
            )

            ax.legend()

            # ---------------- Bottom panel ----------------
            ax = axes[1]

            t = result["t"]
            v = result["soma"]

            ax.plot(t, v, color=colors["soma"], lw=1)

            spike_times = []

            for i in range(1, len(v)):
                if v[i] > 0 and v[i-1] <= 0:
                    spike_times.append(t[i])

            if spike_times:

                ax.scatter(
                    spike_times,
                    [25]*len(spike_times),
                    marker="v",
                    color="red",
                    s=50
                )

                for st in spike_times:
                    ax.axvline(st, color="red", alpha=0.3)

            stim_start = stim_delay.value
            stim_end = stim_delay.value + stim_dur.value

            ax.axvspan(
                stim_start,
                stim_end,
                alpha=0.1,
                color="green",
                label="Stimulus"
            )

            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Soma Voltage (mV)")
            ax.legend()

            plt.tight_layout()
            plt.show()

            print(f"Spikes: {len(spike_times)}")
            print(f"Soma Voltage Range: {min(v):.2f} to {max(v):.2f} mV")

        except Exception as e:
            print("Error:", e)
            import traceback
            traceback.print_exc()


run_btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HBox([stim_loc, Ifull_slider]),
    widgets.HBox([stim_delay, stim_dur, tstop_slider]),
    run_btn,
    out
])

display(ui)

## Channel Distribution

| Compartment | Channels |
|---|---|
| **Soma** | pas, nafast, kfasttab, kslowtab, kA, kca3, lcafixed, cad |
| **Glomerulus** | pas, kslowtab, lcafixed, cad |
| **Primary dendrite** | pas, nafast, kfasttab, kslowtab, lcafixed, cad |
| **Secondary dendrite** | pas, kfasttab, nafast |

## Notes

- The original model uses `FUNCTION_TABLE` for kfasttab and kslowtab, loaded from external data files. This notebook replaces those with analytic TABLE-based implementations for compatibility with NEURON 9.
- Reference: Bhalla US & Bower JM (1993) Exploring parameter space in detailed single neuron models. *J. Neurophysiol.* 69:1948-1983.
- ModelDB entry: [2487](https://senselab.med.yale.edu/ModelDB/ShowModel?model=2487)

## NEW code

In [ ]:
# =====================================================================
# CELL BUILDERS (Functional Approach)
# =====================================================================

def create_GC(name):
    """Simple single-compartment Granule Cell using standard Hodgkin-Huxley."""
    sec = h.Section(name=name)
    sec.L = sec.diam = 15
    sec.insert('pas')
    sec.insert('hh')
    sec.g_pas = 0.0001
    sec.e_pas = -65
    return sec

def build_mit4(n_secondary=2):
    """Build and return the MIT4 mitral cell model with multiple secondary dendrites."""
    # ---------------- Parameters ----------------
    Atotal = 100000.0
    Len = 100.0
    RM = 100000.0
    Erest = -65.0
    p, q, r = 0.051, 0.084, 0.328
    gpg = 5.86e-5
    gsp = 5.47e-5
    gsd = 1.94e-4

    # ---------------- Create sections ----------------
    sections = {}
    for name in ["soma", "glom", "prim", "s2p", "p2g"]:
        sections[name] = h.Section(name=name)

    sections["dend"] = []
    sections["s2d"] = []

    for i in range(n_secondary):
        sections["dend"].append(h.Section(name=f"dend_{i}"))
        sections["s2d"].append(h.Section(name=f"s2d_{i}"))

    soma = sections["soma"]
    glom = sections["glom"]
    prim = sections["prim"]
    s2p = sections["s2p"]
    p2g = sections["p2g"]

    # ---------------- Topology ----------------
    s2p.connect(soma(0), 0)
    prim.connect(s2p(1), 0)
    p2g.connect(prim(1), 0)
    glom.connect(p2g(1), 0)

    for s2d, dend in zip(sections["s2d"], sections["dend"]):
        s2d.connect(soma(1), 0)
        dend.connect(s2d(1), 0)

    for sec in sections["s2d"]:
        sec.L = 1
        sec.diam = 1

    for sec in [s2p, p2g]:
        sec.L = 1
        sec.diam = 1

    # ---------------- Areas ----------------
    Asoma = p * Atotal
    Aglom = q * Atotal
    Aprim = r * Atotal
    Adend = Atotal - Asoma - Aglom - Aprim

    # Split dendrite area equally across branches
    area_each = Adend / n_secondary

    # ---------------- Helper functions ----------------
    def set_size(sec, area):
        sec.diam = area / (np.pi * Len)
        sec.L = Len

    def set_ra(sec, g):
        sec.Ra = (np.pi * 1e4) / (4 * Atotal) * (1.0 / g)

    def setup(sec, area, channels):
        sec.Ra = 1e-7
        set_size(sec, area)
        for ch in channels:
            sec.insert(ch)
        sec.e_pas = Erest
        sec.g_pas = 1.0 / RM

    # ---------------- Soma ----------------
    setup(soma, Asoma, ["pas", "nafast", "kfasttab", "kslowtab", "kA", "kca3", "lcafixed", "cad"])
    soma.gnabar_nafast = 0.1532
    soma.gkbar_kfasttab = 0.1956
    soma.gkbar_kslowtab = 0.0028
    soma.gkbar_kA = 0.00587
    soma.gkbar_kca3 = 0.0142
    soma.gcabar_lcafixed = 0.0040
    soma.depth_cad = 8

    # ---------------- Glomerulus ----------------
    setup(glom, Aglom, ["pas", "kslowtab", "lcafixed", "cad"])
    glom.gkbar_kslowtab = 0.020
    glom.gcabar_lcafixed = 0.0095
    glom.depth_cad = 8

    # ---------------- Primary dendrite ----------------
    setup(prim, Aprim, ["pas", "nafast", "kfasttab", "kslowtab", "lcafixed", "cad"])
    prim.gkbar_kfasttab = 0.00123
    prim.gnabar_nafast = 0.00134
    prim.gkbar_kslowtab = 0.00174
    prim.gcabar_lcafixed = 0.0022
    prim.depth_cad = 8

    # ---------------- Secondary dendrites ----------------
    for dend in sections["dend"]:
        setup(dend, area_each, ["pas", "kfasttab", "nafast"])
        dend.gkbar_kfasttab = 0.0330
        dend.gnabar_nafast = 0.0226

    # ---------------- Axial resistances ----------------
    for sec in sections["s2d"]:
        # Scale conductance by the number of branches so total resistance is biologically preserved
        set_ra(sec, gsd / n_secondary)

    set_ra(s2p, gsp)
    set_ra(p2g, gpg)

    # ---------------- Reversal potentials ----------------
    for sec in h.allsec():
        if sec.has_membrane("ca_ion"):
            sec.eca = 70; sec.cai = 1e-5; sec.cao = 2
        if sec.has_membrane("na_ion"):
            sec.ena = 45
        if sec.has_membrane("k_ion"):
            sec.ek = -70

    return sections

# =====================================================================
# NETWORK WIRING & SIMULATION
# =====================================================================

def reciprocal_network(Ifull=1.6, stim_location='Soma', stim_delay=50, stim_dur=100, tstop=200,
                       n_secondary=2, gcs_per_dend=1):

    mitral_cell = build_mit4(n_secondary=n_secondary)

    # Keep track of objects so they aren't garbage collected
    gcs = {}
    syns = []
    ncs = []

    # ---------------- Setup Reciprocal Synapses ----------------
    for i, dend in enumerate(mitral_cell['dend']):

        # Randomize positions if more than 1 GC, else perfectly center it
        if gcs_per_dend == 1:
            positions = [0.5]
        else:
            positions = [random.uniform(0.1, 0.9) for _ in range(gcs_per_dend)]

        for j, pos in enumerate(positions):
            gc_name = f"GC_d{i}_g{j}"
            gc = create_GC(gc_name)
            gcs[gc_name] = gc

            # Exc Synapse (Mitral -> GC)
            ampa = h.ExpSyn(gc(0.5))
            ampa.e = 0
            ampa.tau = 2

            nc_exc = h.NetCon(dend(pos)._ref_v, ampa, sec=dend)
            nc_exc.weight[0] = 0.5

            # Inh Synapse (GC -> Mitral)
            gaba = h.ExpSyn(dend(pos))
            gaba.e = -75
            gaba.tau = 2

            nc_inh = h.NetCon(gc(0.5)._ref_v, gaba, sec=gc)
            nc_inh.weight[0] = 2.5

            # General params
            for nc in [nc_exc, nc_inh]:
                nc.delay = 1.0
                nc.threshold = -20

            syns.extend([ampa, gaba])
            ncs.extend([nc_exc, nc_inh])

    # ---------------- Stimulus Part ----------------
    alphas = 1.37
    alphag = 1.85
    Atotal = 100000.0
    injcurrdens = Ifull / 100072.0

    stim_objs = [] # Keep alive

    if stim_location in ['Soma', 'Both']:
        stim = h.IClamp(mitral_cell['soma'](0.5))
        stim.delay = stim_delay; stim.dur = stim_dur
        stim.amp = alphas * injcurrdens * Atotal
        stim_objs.append(stim)

    if stim_location in ['Glomerulus', 'Both']:
        stim = h.IClamp(mitral_cell['glom'](0.5))
        stim.delay = stim_delay; stim.dur = stim_dur
        stim.amp = alphag * injcurrdens * Atotal
        stim_objs.append(stim)

    # ---------------- Recorders ----------------
    rec = {}
    rec['t'] = h.Vector().record(h._ref_t)
    rec['soma'] = h.Vector().record(mitral_cell['soma'](0.5)._ref_v)
    rec['glom'] = h.Vector().record(mitral_cell['glom'](0.5)._ref_v)
    rec['prim'] = h.Vector().record(mitral_cell['prim'](0.5)._ref_v)

    for i, dend in enumerate(mitral_cell['dend']):
        rec[f'dend_{i}'] = h.Vector().record(dend(0.5)._ref_v)

    for gc_name, gc in gcs.items():
        rec[gc_name] = h.Vector().record(gc(0.5)._ref_v)

    # ---------------- Initialize and Run ----------------
    h.dt = 0.025
    h.finitialize(-65)

    while h.t < tstop:
        h.fadvance()

    # We return the data, but ALSO the gc names so the plotter knows what to look for!
    return {
        'data': {k: np.array(v) for k, v in rec.items()},
        'dend_names': [f'dend_{i}' for i in range(n_secondary)],
        'gc_names': list(gcs.keys())
    }


# =====================================================================
# INTERACTIVE CONTROLS & PLOTTING
# =====================================================================

stim_loc = widgets.Dropdown(options=['Soma', 'Glomerulus', 'Both'], value='Soma', description='Stimulus at:')
Ifull_slider = widgets.FloatSlider(value=1.6, min=0.0, max=3.0, step=0.1, description='Ifull (nA):', readout_format='.2f')
stim_dur = widgets.FloatSlider(value=100, min=5, max=500, step=5, description='Duration (ms):')
stim_delay = widgets.FloatSlider(value=50, min=0, max=200, step=5, description='Delay (ms):')
tstop_slider = widgets.FloatSlider(value=200, min=20, max=500, step=10, description='Tstop (ms):')

run_btn = widgets.Button(description='Run Simulation', button_style='primary', layout=widgets.Layout(width='200px'))
out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        print("Running flexible multi-dendrite simulation...\n")
        try:
            # You can change n_secondary or gcs_per_dend here!
            network_out = reciprocal_network(
                Ifull=Ifull_slider.value,
                stim_location=stim_loc.value,
                stim_delay=stim_delay.value,
                stim_dur=stim_dur.value,
                tstop=tstop_slider.value,
                n_secondary=2,     # Adjust as needed
                gcs_per_dend=1     # Adjust as needed
            )

            result = network_out['data']
            dend_names = network_out['dend_names']
            gc_names = network_out['gc_names']

            fig, axes = plt.subplots(2, 1, figsize=(11,7), sharex=True)

            # Core colors
            colors = {'soma': '#1f77b4', 'glom': '#ff7f0e', 'prim': '#2ca02c'}
            # Generate colormap for variable dendrites and GCs
            cmap_dends = plt.cm.get_cmap('Reds', len(dend_names) + 2)
            cmap_gcs = plt.cm.get_cmap('Purples', len(gc_names) + 2)

            # TOP PANEL ####################################################
            ax = axes[0]

            # Plot Core
            for name in ['soma', 'glom', 'prim']:
                ax.plot(result['t'], result[name], label=name, color=colors[name], lw=1)

            # Plot all Dendrites
            for i, d_name in enumerate(dend_names):
                ax.plot(result['t'], result[d_name], label=d_name, color=cmap_dends(i+2), lw=1)

            # Plot all GCs
            for i, g_name in enumerate(gc_names):
                ax.plot(result['t'], result[g_name], label=g_name, color=cmap_gcs(i+2), lw=1, alpha=0.7)

            ax.axhline(0, color='gray', ls=':', lw=0.5)
            ax.set_ylabel("Voltage (mV)")
            ax.set_title(f"Reciprocal Mitral–Granule Network ({len(dend_names)} Dends, {len(gc_names)} GCs)\nIfull={Ifull_slider.value:.1f} nA | {stim_loc.value}")
            ax.legend(loc='upper right', ncol=4, fontsize=8)

            # BOTTOM PANEL ####################################################
            ax = axes[1]
            t = result['t']
            soma = result['soma']

            ax.plot(t, soma, color=colors['soma'], lw=1.5, label='Mitral Soma')

            for i, g_name in enumerate(gc_names):
                ax.plot(t, result[g_name], color=cmap_gcs(i+2), lw=1.2, label=g_name)

            # Detect mitral spikes
            spike_times = []
            for i in range(1, len(soma)):
                if soma[i] > 0 and soma[i-1] <= 0:
                    spike_times.append(t[i])

            if spike_times:
                ax.scatter(spike_times, [25]*len(spike_times), color='red', marker='v', s=50, zorder=10, label='Mitral spikes')
                for st in spike_times:
                    ax.axvline(st, color='red', alpha=0.3, lw=0.6)

            stim_start = stim_delay.value
            stim_end = stim_delay.value + stim_dur.value
            ax.axvspan(stim_start, stim_end, color='green', alpha=0.08, label='Stimulus')

            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Voltage (mV)")
            ax.legend(fontsize=8, loc='upper right', ncol=3)

            plt.tight_layout()
            plt.show()

            # Summary ####################################################
            print(f"Mitral spikes : {len(spike_times)}")
            print(f"Soma V range  : [{np.min(soma):.1f}, {np.max(soma):.1f}] mV")

            if len(spike_times) > 1:
                isi = np.diff(spike_times)
                print(f"Mean ISI      : {np.mean(isi):.2f} ms")
                print(f"Frequency     : {1000/np.mean(isi):.2f} Hz")

        except Exception as e:
            print("Error:", e)
            import traceback
            traceback.print_exc()

run_btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HBox([stim_loc, Ifull_slider]),
    widgets.HBox([stim_delay, stim_dur, tstop_slider]),
    run_btn,
    out
])

display(ui)

In [36]:
from neuron import h

cell = build_mit4(2)

h.topology()


|-|       __nrnsec_00000178ad13f100(0-1)
|-|       soma(0-1)
   `|       s2d_0(0-1)
     `|       dend_0(0-1)
   `|       s2d_1(0-1)
     `|       dend_1(0-1)
 `|       s2p(0-1)
   `|       prim(0-1)
     `|       p2g(0-1)
       `|       glom(0-1)



1.0

## New New code

In [ ]:
# =====================================================================
# CELL BUILDERS (Functional Approach)
# =====================================================================

def create_GC(name):
    """Simple single-compartment Granule Cell using standard Hodgkin-Huxley."""
    sec = h.Section(name=name)
    sec.L = sec.diam = 15
    sec.insert('pas')
    sec.insert('hh')
    sec.g_pas = 0.0001
    sec.e_pas = -65
    return sec

def build_mit4(n_secondary=2):
    """Build and return the MIT4 mitral cell model with multiple secondary dendrites."""
    # ---------------- Parameters ----------------
    Atotal = 100000.0
    Len = 100.0
    RM = 100000.0
    Erest = -65.0
    p, q, r = 0.051, 0.084, 0.328
    gpg = 5.86e-5
    gsp = 5.47e-5
    gsd = 1.94e-4

    # ---------------- Create sections ----------------
    sections = {}
    for name in ["soma", "glom", "prim", "s2p", "p2g"]:
        sections[name] = h.Section(name=name)

    sections["dend"] = []
    sections["s2d"] = []

    for i in range(n_secondary):
        sections["dend"].append(h.Section(name=f"dend_{i}"))
        sections["s2d"].append(h.Section(name=f"s2d_{i}"))

    soma = sections["soma"]
    glom = sections["glom"]
    prim = sections["prim"]
    s2p = sections["s2p"]
    p2g = sections["p2g"]

    # ---------------- Topology ----------------
    s2p.connect(soma(0), 0)
    prim.connect(s2p(1), 0)
    p2g.connect(prim(1), 0)
    glom.connect(p2g(1), 0)

    for s2d, dend in zip(sections["s2d"], sections["dend"]):
        s2d.connect(soma(1), 0)
        dend.connect(s2d(1), 0)

    for sec in sections["s2d"]:
        sec.L = 1
        sec.diam = 1

    for sec in [s2p, p2g]:
        sec.L = 1
        sec.diam = 1

    # ---------------- Areas ----------------
    Asoma = p * Atotal
    Aglom = q * Atotal
    Aprim = r * Atotal
    Adend = Atotal - Asoma - Aglom - Aprim

    # Split dendrite area equally across branches
    area_each = Adend / max(1, n_secondary)

    # ---------------- Helper functions ----------------
    def set_size(sec, area):
        sec.diam = area / (np.pi * Len)
        sec.L = Len

    def set_ra(sec, g):
        sec.Ra = (np.pi * 1e4) / (4 * Atotal) * (1.0 / g)

    def setup(sec, area, channels):
        sec.Ra = 1e-7
        set_size(sec, area)
        for ch in channels:
            sec.insert(ch)
        sec.e_pas = Erest
        sec.g_pas = 1.0 / RM

    # ---------------- Soma ----------------
    setup(soma, Asoma, ["pas", "nafast", "kfasttab", "kslowtab", "kA", "kca3", "lcafixed", "cad"])
    soma.gnabar_nafast = 0.1532
    soma.gkbar_kfasttab = 0.1956
    soma.gkbar_kslowtab = 0.0028
    soma.gkbar_kA = 0.00587
    soma.gkbar_kca3 = 0.0142
    soma.gcabar_lcafixed = 0.0040
    soma.depth_cad = 8

    # ---------------- Glomerulus ----------------
    setup(glom, Aglom, ["pas", "kslowtab", "lcafixed", "cad"])
    glom.gkbar_kslowtab = 0.020
    glom.gcabar_lcafixed = 0.0095
    glom.depth_cad = 8

    # ---------------- Primary dendrite ----------------
    setup(prim, Aprim, ["pas", "nafast", "kfasttab", "kslowtab", "lcafixed", "cad"])
    prim.gkbar_kfasttab = 0.00123
    prim.gnabar_nafast = 0.00134
    prim.gkbar_kslowtab = 0.00174
    prim.gcabar_lcafixed = 0.0022
    prim.depth_cad = 8

    # ---------------- Secondary dendrites ----------------
    for dend in sections["dend"]:
        setup(dend, area_each, ["pas", "kfasttab", "nafast"])
        dend.gkbar_kfasttab = 0.0330
        dend.gnabar_nafast = 0.0226

    # ---------------- Axial resistances ----------------
    for sec in sections["s2d"]:
        # Scale conductance by the number of branches so total resistance is biologically preserved
        set_ra(sec, gsd / max(1, n_secondary))

    set_ra(s2p, gsp)
    set_ra(p2g, gpg)

    # ---------------- Reversal potentials ----------------
    for sec in h.allsec():
        if sec.has_membrane("ca_ion"):
            sec.eca = 70; sec.cai = 1e-5; sec.cao = 2
        if sec.has_membrane("na_ion"):
            sec.ena = 45
        if sec.has_membrane("k_ion"):
            sec.ek = -70

    return sections

# =====================================================================
# NETWORK WIRING & SIMULATION
# =====================================================================

def reciprocal_network(Ifull=1.6, stim_location='Soma', stim_delay=50, stim_dur=100, tstop=200,
                       n_secondary=2, gcs_per_dend=1):

    mitral_cell = build_mit4(n_secondary=n_secondary)

    # Keep track of objects so they aren't garbage collected
    gcs = {}
    gc_info = {} # Track position for plotting
    syns = []
    ncs = []

    # ---------------- Setup Reciprocal Synapses ----------------
    for i, dend in enumerate(mitral_cell['dend']):

        # Randomize positions if more than 1 GC, else perfectly center it
        if gcs_per_dend == 1:
            positions = [0.5]
        elif gcs_per_dend > 1:
            positions = [random.uniform(0.1, 0.9) for _ in range(gcs_per_dend)]
        else:
            positions = []

        for j, pos in enumerate(positions):
            gc_name = f"GC_d{i}_g{j}"
            gc = create_GC(gc_name)
            gcs[gc_name] = gc
            gc_info[gc_name] = pos  # Save the connection location

            # Exc Synapse (Mitral -> GC)
            ampa = h.ExpSyn(gc(0.5))
            ampa.e = 0
            ampa.tau = 2

            nc_exc = h.NetCon(dend(pos)._ref_v, ampa, sec=dend)
            nc_exc.weight[0] = 0.5

            # Inh Synapse (GC -> Mitral)
            gaba = h.ExpSyn(dend(pos))
            gaba.e = -75
            gaba.tau = 2

            nc_inh = h.NetCon(gc(0.5)._ref_v, gaba, sec=gc)
            nc_inh.weight[0] = 2.5

            # General params
            for nc in [nc_exc, nc_inh]:
                nc.delay = 1.0
                nc.threshold = -20

            syns.extend([ampa, gaba])
            ncs.extend([nc_exc, nc_inh])

    # ---------------- Stimulus Part ----------------
    alphas = 1.37
    alphag = 1.85
    Atotal = 100000.0
    injcurrdens = Ifull / 100072.0

    stim_objs = [] # Keep alive

    if stim_location in ['Soma', 'Both']:
        stim = h.IClamp(mitral_cell['soma'](0.5))
        stim.delay = stim_delay; stim.dur = stim_dur
        stim.amp = alphas * injcurrdens * Atotal
        stim_objs.append(stim)

    if stim_location in ['Glomerulus', 'Both']:
        stim = h.IClamp(mitral_cell['glom'](0.5))
        stim.delay = stim_delay; stim.dur = stim_dur
        stim.amp = alphag * injcurrdens * Atotal
        stim_objs.append(stim)

    # ---------------- Recorders ----------------
    rec = {}
    rec['t'] = h.Vector().record(h._ref_t)
    rec['soma'] = h.Vector().record(mitral_cell['soma'](0.5)._ref_v)
    rec['glom'] = h.Vector().record(mitral_cell['glom'](0.5)._ref_v)
    rec['prim'] = h.Vector().record(mitral_cell['prim'](0.5)._ref_v)

    for i, dend in enumerate(mitral_cell['dend']):
        rec[f'dend_{i}'] = h.Vector().record(dend(0.5)._ref_v)

    for gc_name, gc in gcs.items():
        rec[gc_name] = h.Vector().record(gc(0.5)._ref_v)

    # ---------------- Initialize and Run ----------------
    h.dt = 0.025
    h.finitialize(-65)

    while h.t < tstop:
        h.fadvance()

    # Return data alongside names to easily identify components in plotting
    return {
        'data': {k: np.array(v) for k, v in rec.items()},
        'dend_names': [f'dend_{i}' for i in range(n_secondary)],
        'gc_names': list(gcs.keys()),
        'gc_info': gc_info
    }


# =====================================================================
# INTERACTIVE CONTROLS & PLOTTING
# =====================================================================

# Stimulation UI Controls
stim_loc = widgets.Dropdown(options=['Soma', 'Glomerulus', 'Both'], value='Soma', description='Stimulus at:')
Ifull_slider = widgets.FloatSlider(value=1.6, min=0.0, max=3.0, step=0.1, description='Ifull (nA):', readout_format='.2f')
stim_dur = widgets.FloatSlider(value=100, min=5, max=500, step=5, description='Duration (ms):')
stim_delay = widgets.FloatSlider(value=50, min=0, max=200, step=5, description='Delay (ms):')
tstop_slider = widgets.FloatSlider(value=200, min=20, max=500, step=10, description='Tstop (ms):')

# Network Topology UI Controls
n_secondary_slider = widgets.IntSlider(value=2, min=1, max=5, description='Dendrites:')
gcs_per_dend_slider = widgets.IntSlider(value=1, min=0, max=5, description='GCs/Dend:')

run_btn = widgets.Button(description='Run Simulation', button_style='primary', layout=widgets.Layout(width='200px'))
out = widgets.Output()

def on_run(b):
    with out:
        clear_output(wait=True)
        print("Running flexible multi-dendrite simulation...\n")
        try:
            n_sec = n_secondary_slider.value
            n_gcs = gcs_per_dend_slider.value

            network_out = reciprocal_network(
                Ifull=Ifull_slider.value,
                stim_location=stim_loc.value,
                stim_delay=stim_delay.value,
                stim_dur=stim_dur.value,
                tstop=tstop_slider.value,
                n_secondary=n_sec,
                gcs_per_dend=n_gcs
            )

            result = network_out['data']
            dend_names = network_out['dend_names']
            gc_names = network_out['gc_names']
            gc_info = network_out['gc_info']
            t = result['t']

            # Dynamic plotting:
            # 1 Core + 1 Combined Dends/GCs + (N individual Dends/GCs) + 1 Soma Spikes
            n_subplots = 3 + n_sec
            fig, axes = plt.subplots(n_subplots, 1, figsize=(12, 3.5 * n_subplots), sharex=True)

            colors = {'soma': '#1f77b4', 'glom': '#ff7f0e', 'prim': '#2ca02c'}
            cmap_dends = plt.get_cmap('Reds')
            cmap_gcs = plt.get_cmap('Purples')

            # --- PANEL 0: Core Mitral Components ---
            ax_core = axes[0]
            for name in ['soma', 'glom', 'prim']:
                ax_core.plot(t, result[name], label=name, color=colors[name], lw=1.5)
            ax_core.axhline(0, color='gray', ls=':', lw=0.5)
            ax_core.set_ylabel("Voltage (mV)")
            ax_core.set_title(f"Core Mitral Activity | Ifull={Ifull_slider.value:.1f} nA | {stim_loc.value}")
            ax_core.legend(loc='upper right', ncol=3, fontsize=9)

            # --- PANEL 1: All Dendrites & Granule Cells COMBINED ---
            ax_combined = axes[1]

            # Plot Dendrites in Combined Panel
            for i, d_name in enumerate(dend_names):
                c_val = 0.4 + 0.6 * (i / max(1, len(dend_names) - 1)) if len(dend_names) > 1 else 0.7
                ax_combined.plot(t, result[d_name], color=cmap_dends(c_val), lw=1.5, label=f'Mitral {d_name}')

            # Plot Granule Cells in Combined Panel with location detail
            for k, g_name in enumerate(gc_names):
                c_val = 0.4 + 0.6 * (k / max(1, len(gc_names) - 1)) if len(gc_names) > 1 else 0.7
                pos = gc_info[g_name]
                label_str = f"{g_name} (pos={pos:.2f})"
                ax_combined.plot(t, result[g_name], color=cmap_gcs(c_val), lw=1, alpha=0.8, label=label_str)

            ax_combined.axhline(0, color='gray', ls=':', lw=0.5)
            ax_combined.set_ylabel("Voltage (mV)")
            ax_combined.set_title("Combined Lateral Dendrite & Granule Cell Activity")

            # Dynamically adjust legend columns based on number of items so it doesn't overrun
            n_legend_items = len(dend_names) + len(gc_names)
            cols = min(5, (n_legend_items // 3) + 1)
            ax_combined.legend(loc='upper right', ncol=cols, fontsize=7)

            # --- PANELS 2 to (2 + n_sec - 1): INDIVIDUAL Dendrite & GC pairings ---
            for i in range(n_sec):
                ax_indiv = axes[2 + i]
                d_name = dend_names[i]

                # Fetch consistent color for this dendrite from cmap
                c_val_d = 0.4 + 0.6 * (i / max(1, len(dend_names) - 1)) if len(dend_names) > 1 else 0.7
                ax_indiv.plot(t, result[d_name], color=cmap_dends(c_val_d), lw=1.5, label=f'Mitral {d_name}')

                # Find and plot all GCs attached specifically to THIS dendrite
                for k, g_name in enumerate(gc_names):
                    if f'_d{i}_' in g_name:
                        # Match the color index identically to how it was assigned in the combined plot
                        c_val_g = 0.4 + 0.6 * (k / max(1, len(gc_names) - 1)) if len(gc_names) > 1 else 0.7
                        pos = gc_info[g_name]
                        label_str = f"{g_name} (pos={pos:.2f})"
                        ax_indiv.plot(t, result[g_name], color=cmap_gcs(c_val_g), lw=1.2, alpha=0.9, label=label_str)

                ax_indiv.axhline(0, color='gray', ls=':', lw=0.5)
                ax_indiv.set_ylabel("Voltage (mV)")
                ax_indiv.set_title(f"Isolated Subnetwork: {d_name} & Attached Granule Cells")
                ax_indiv.legend(loc='upper right', ncol=max(1, n_gcs + 1), fontsize=8)


            # --- BOTTOM PANEL: Mitral Soma & Spike Detection ---
            ax_soma = axes[-1]
            soma = result['soma']
            ax_soma.plot(t, soma, color=colors['soma'], lw=1.5, label='Mitral Soma')

            # Detect mitral spikes
            spike_times = []
            for i in range(1, len(soma)):
                if soma[i] > 0 and soma[i-1] <= 0:
                    spike_times.append(t[i])

            if spike_times:
                ax_soma.scatter(spike_times, [25]*len(spike_times), color='red', marker='v', s=50, zorder=10, label='Mitral spikes')
                for st in spike_times:
                    ax_soma.axvline(st, color='red', alpha=0.3, lw=0.6)

            ax_soma.set_xlabel("Time (ms)")
            ax_soma.set_ylabel("Voltage (mV)")
            ax_soma.set_title("Mitral Soma Spike Readout")
            ax_soma.legend(fontsize=9, loc='upper right')

            # Highlight Stimulus Area on ALL subplots
            stim_start = stim_delay.value
            stim_end = stim_delay.value + stim_dur.value
            for ax in axes:
                ax.axvspan(stim_start, stim_end, color='green', alpha=0.08, label='Stimulus' if ax == axes[-1] else "")

            plt.tight_layout()
            plt.show()

            # --- Summary ---
            print(f"Network built with {n_sec} lateral dendrites and {n_gcs} Granule Cells per dendrite (Total GCs: {n_sec * n_gcs}).")
            print(f"Mitral spikes : {len(spike_times)}")
            print(f"Soma V range  : [{np.min(soma):.1f}, {np.max(soma):.1f}] mV")

            if len(spike_times) > 1:
                isi = np.diff(spike_times)
                print(f"Mean ISI      : {np.mean(isi):.2f} ms")
                print(f"Frequency     : {1000/np.mean(isi):.2f} Hz")

        except Exception as e:
            print("Error:", e)
            import traceback
            traceback.print_exc()

run_btn.on_click(on_run)

ui = widgets.VBox([
    widgets.HBox([stim_loc, Ifull_slider]),
    widgets.HBox([stim_delay, stim_dur, tstop_slider]),
    widgets.HBox([n_secondary_slider, gcs_per_dend_slider]),
    run_btn,
    out
])

display(ui)